# Meet the environment

Step 2 of phase 3. You're going to poke at the fishing game from Python by hand, see exactly what a player sees, and then write `encode()`: the function that turns that into a list of numbers a neural network can read.

Run each cell with **Shift+Enter**. The game here is the real one from `src/game`, played through `training/bridge.ts`.

In [1]:
from pprint import pprint

from fishing_env import FishingEnv, cast, REEL, WAIT, WIDTH, HEIGHT

env = FishingEnv()
obs, info = env.reset(seed=1)
pprint({k: v for k, v in obs.items() if k != "fish"})
print(len(obs["fish"]), "fish")

{'catches': 0,
 'freshness': 0,
 'hook': None,
 'line': 'idle',
 'nibbleLeft': 0,
 'reason': 'idle',
 'rod': {'x': 178, 'y': 8},
 'score': 0,
 'tick': 0,
 'ticksLeft': 7200}
30 fish


That's one **observation**: what the player sees at a decision point. `reason` says why it's being asked (`idle`, `landed`, `nibble` or `check`). Now the fish. Each one has a position (its center, in pixels), a velocity in pixels per tick, a species, how many points it's worth, and whether it's scared right now.

In [2]:
for f in obs["fish"][:5]:
    print(f)

{'id': 0, 'x': 234.34571849554777, 'y': 132.10161643102765, 'vx': -0.3423165578395128, 'vy': 0, 'species': 4, 'points': 3, 'scared': False}
{'id': 1, 'x': 75.16544634476304, 'y': 127.24875606968999, 'vx': 0.4059142398810945, 'vy': 0, 'species': 0, 'points': 1, 'scared': False}
{'id': 2, 'x': 122.86437583342195, 'y': 48.46949354559183, 'vx': 0.14347363624721765, 'vy': 0, 'species': 3, 'points': 1, 'scared': False}
{'id': 3, 'x': 240.1600788012147, 'y': 52.44031571596861, 'vx': -0.25809555976185944, 'vy': 0, 'species': 4, 'points': 3, 'scared': False}
{'id': 4, 'x': 272.6219255141914, 'y': 69.59397791698575, 'vx': 0.5496773572778328, 'vy': 0, 'species': 1, 'points': 2, 'scared': False}


Take a few actions and watch what changes. `info["teacher"]` is what the hand-written heuristic would do at this moment, which is useful for comparison now and essential for behavior cloning later.

In [3]:
print("the heuristic would:", info["teacher"])

obs, reward, done, _, info = env.step(cast(160, 90))   # cast to the middle of the bowl
print(obs["reason"], obs["line"], "hook at", obs["hook"], "reward", reward)

for _ in range(5):
    obs, reward, done, _, info = env.step(WAIT)
    print(obs["tick"], obs["reason"], obs["line"], "freshness", round(obs["freshness"], 2), "reward", reward)

the heuristic would: {'type': 'cast', 'x': 168.69154801391338, 'y': 64.01532858610153}
landed waiting hook at {'x': 160, 'y': 90} reward 0.0
71 check waiting freshness 0.92 reward 0.0
101 check waiting freshness 0.83 reward 0.0
131 check waiting freshness 0.75 reward 0.0
161 check waiting freshness 0.67 reward 0.0
191 check waiting freshness 0.58 reward 0.0


## Your turn: `encode(obs)`

A network can't read a dict. It needs a **fixed-length list of numbers**, the same length every time, and it learns best when those numbers are roughly between -1 and 1. Deciding what goes in that list is called *feature engineering*, and it matters as much as the network itself. If something isn't in the vector, the model can't know it.

Some questions to decide:

- **The line:** `obs["line"]` is a word (`idle`, `casting`, `waiting`, `nibble`, `reeling`). Networks want numbers, and the usual trick is *one-hot*: five slots, one of them 1.0 and the rest 0.0.
- **Positions:** pixels run 0..320 and 0..180. Divide by `WIDTH` and `HEIGHT` to scale them. Should fish be absolute positions, or **relative to the hook** (or the rod, while the line is in)? Relative is often easier to learn from.
- **The fish:** there are 30, but the list order isn't meaningful, and fish come and go. A common approach is the **nearest K fish** (say 5 or 8), sorted by distance, with a few numbers each. If there are fewer than K, pad with zeros.
- **Other signals:** `freshness`, `nibbleLeft` (up to 24), `ticksLeft` (up to 7200). Scale each to 0..1.

There's no single right answer. Start simple. You can come back and change this once you see how training goes.

In [10]:
LINE_STATES = ["idle", "casting", "waiting", "nibble", "reeling"]
K = 5  # nearest fish to include


def encode(obs):
    """Turn an observation dict into a fixed-length list of floats, roughly in -1..1."""
    features = []
    # the line state, one-hot (5 numbers)
    for state in LINE_STATES:
        features.append(1.0 if obs["line"] == state else 0.0)

    # the hook position, scaled (2 numbers)
    if obs["hook"] is None:
        features.append(0.0)
        features.append(0.0)
    else:
        features.append(obs["hook"]["x"] / WIDTH)
        features.append(obs["hook"]["y"] / HEIGHT)

    # freshness and the two timers (3 numbers)
    features.append(obs["freshness"])
    features.append(obs["nibbleLeft"] / 24)
    features.append(obs["ticksLeft"] / 7200)

    # the K nearest fish, measured from the hook (or the rod while the line is in), 6 numbers each
    start = obs["hook"] or obs["rod"]
    nearest = sorted(
        obs["fish"],
        key=lambda f: (f["x"] - start["x"]) ** 2 + (f["y"] - start["y"]) ** 2,
    )[:K]
    for f in nearest:
        features.append((f["x"] - start["x"]) / WIDTH)
        features.append((f["y"] - start["y"]) / HEIGHT)
        features.append(f["vx"] / 2)  # a frightened fish can hit about 2.3 px/tick
        features.append(f["vy"])
        features.append(f["points"] / 5)
        features.append(1.0 if f["scared"] else 0.0)
    for _ in range(K - len(nearest)):  # pad if fewer than K fish are swimming
        features += [0.0] * 6
        
    return features

In [7]:
obs, info = env.reset(seed=1)

obs, reward, done, _, info = env.step(cast(160, 90))   # cast to the middle of the bowl
print("just landed:", obs["reason"], obs["line"], "freshness", obs["freshness"])

for _ in range(15):
    obs, reward, done, _, info = env.step(WAIT)
    print("tick", obs["tick"], obs["line"], "freshness", round(obs["freshness"], 3))
    if obs["line"] != "waiting":
        break

just landed: landed waiting freshness 1
tick 71 waiting freshness 0.917
tick 101 waiting freshness 0.833
tick 131 waiting freshness 0.75
tick 161 waiting freshness 0.667
tick 191 waiting freshness 0.583
tick 221 waiting freshness 0.5
tick 251 waiting freshness 0.417
tick 281 waiting freshness 0.333
tick 311 waiting freshness 0.25
tick 341 waiting freshness 0.167
tick 371 waiting freshness 0.083
tick 401 waiting freshness 0
tick 431 waiting freshness 0
tick 461 waiting freshness 0
tick 491 waiting freshness 0


## Check it

This plays one full run with the heuristic's actions, encodes every observation, and checks two things: the length never changes, and the values stay in a sensible range. When it prints `looks good`, `encode()` is ready for step 3.

In [11]:
import math

obs, info = env.reset(seed=1)
lengths, lo, hi, n, done = set(), math.inf, -math.inf, 0, False
while not done:
    x = encode(obs)
    lengths.add(len(x))
    lo, hi = min([lo, *x]), max([hi, *x])
    n += 1
    obs, reward, done, _, info = env.step(info["teacher"])

print(f"{n} observations, vector lengths {sorted(lengths)}, values {lo:.2f} .. {hi:.2f}")
assert len(lengths) == 1 and 0 not in lengths, "every vector should have the same, non-zero length"
assert -2 <= lo and hi <= 2, "values look unscaled; divide positions by WIDTH/HEIGHT and so on"
print("looks good")

207 observations, vector lengths [40], values -0.94 .. 1.00
looks good


In [15]:
env.close()